# Retrain 5 seeds × 2 losses and export logits

Event work for the Meta hackathon. It runs the **prior-work** training code (`ebl_fundus`, from the IntelliSys paper)
for five seeds of each loss, keeps every checkpoint, then exports IDRiD and APTOS logits for `energy_eval.py`.

**Before running**, put these in Google Drive under `MyDrive/ebl_fundus/`:
- `idrid.zip`: already there from the paper
- `ebl_fundus-main.zip`: the training code (GitHub → Code → Download ZIP)
- `meta-hackathon.zip`: the event scripts

For APTOS you'll also need a Kaggle API token (see the APTOS step).

Any GPU runtime works (ResNet-18 is small).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/ebl_fundus'

!unzip -q -o {D}/ebl_fundus-main.zip -d /content
!unzip -q -o {D}/meta-hackathon.zip -d /content/meta-hackathon
%cd /content/ebl_fundus-main
!pip install -q -r requirements.txt

Prepare IDRiD exactly as in the original notebook, and reuse the paper's split if it was saved to Drive.

In [ ]:
import os, shutil, sys, zipfile
import pandas as pd

# class_names fix carried over from the original notebook
src = open('src/dataset.py').read()
open('src/dataset.py', 'w').write(src.replace('loaders["class_names"] = class_names',
                                              'loaders["class_names"] = [str(c) for c in class_names]'))

zipfile.ZipFile(f'{D}/idrid.zip').extractall('/content/idrid_raw')
df = pd.read_csv('/content/idrid_raw/idrid_labels.csv')[['id_code', 'diagnosis']].dropna()
df = df.rename(columns={'id_code': 'image_id', 'diagnosis': 'label'}).astype({'label': int})

os.makedirs('data/idrid/images', exist_ok=True)
raw = '/content/idrid_raw/Imagenes/Imagenes'
for f in os.listdir(raw):
    if f.endswith('.jpg') and 'test' not in f.lower():
        shutil.copy(f'{raw}/{f}', 'data/idrid/images')
have = {f[:-4] for f in os.listdir('data/idrid/images')}
df[df.image_id.isin(have)].to_csv('data/idrid/labels.csv', index=False)

if os.path.exists(f'{D}/splits.csv'):
    shutil.copy(f'{D}/splits.csv', 'data/idrid/splits.csv')
    print('reusing the paper split')
else:
    sys.path.insert(0, 'src')
    from dataset import create_splits
    create_splits('data/idrid/labels.csv', 'data/idrid/splits.csv', seed=42)
print(pd.read_csv('data/idrid/splits.csv').split.value_counts().to_string())

Train all 10 runs with the paper's settings. The split stays fixed; only the training seed changes. Every checkpoint is copied to Drive.

In [ ]:
for seed in [42, 43, 44, 45, 46]:
    for loss in ['softmax', 'ebm_margin']:
        !python src/train.py --loss {loss} --margin 1.0 --epochs 10 --lr 1e-4 --batch_size 32 --image_size 224 --seed {seed}

shutil.copytree('runs', f'{D}/runs_event', dirs_exist_ok=True)
shutil.copy('outputs/metrics.csv', f'{D}/metrics_event.csv')
print(pd.read_csv('outputs/metrics.csv')[['loss_type', 'seed', 'test_acc', 'test_macro_f1']].to_string())

APTOS-2019 labelled images, from the official Kaggle competition page:
https://www.kaggle.com/competitions/aptos2019-blindness-detection/data

1. On that page, accept the competition rules (the download is refused until you do).
2. Kaggle → your profile picture → **Settings** → **API** → **Generate New Token**, and copy the token.
3. In Colab, click the key icon (**Secrets**) on the left, add `KAGGLE_API_TOKEN` with that value, and allow notebook access.

An older `kaggle.json` in the Drive folder also works. The download is several GB. To use APTOS images already
in your Drive instead, skip the download lines and point `APTOS_CSV` / `APTOS_DIR` at them (columns `id_code`, `diagnosis`).

In [ ]:
from google.colab import userdata
!pip install -q -U kaggle
try:
    os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')   # Colab Secrets
except Exception:
    !mkdir -p ~/.kaggle && cp {D}/kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

!kaggle competitions download -c aptos2019-blindness-detection -p /content/aptos
!unzip -q -o /content/aptos/aptos2019-blindness-detection.zip "train.csv" "train_images/*" -d /content/aptos

APTOS_CSV, APTOS_DIR = '/content/aptos/train.csv', '/content/aptos/train_images'
print(len(pd.read_csv(APTOS_CSV)), 'labelled APTOS images')

Export logits for every run, then evaluate. The model outputs logits (energy = −logit), so `energy_eval.py` runs without `--energies`.

In [ ]:
!python /content/meta-hackathon/export_logits.py --aptos_csv {APTOS_CSV} --aptos_dir {APTOS_DIR}
!python /content/meta-hackathon/energy_eval.py npz/ebl npz/ce --out energy_eval.csv

shutil.copytree('npz', f'{D}/npz_event', dirs_exist_ok=True)
shutil.copy('energy_eval.csv', D)